# Day 19 · Senses: Smell — What You Breathe While You Wait

**Question.** A rider waiting at the curb breathes the street. How much traffic exhaust is in the air
where Chicago waits for the bus, and when and where is it worst?

**Approach.** Open Air Chicago, the city's network of about 280 air sensors (Chicago Department of
Public Health and UIC School of Public Health, Clarity sensors), reports nitrogen dioxide (NO₂) every
hour. NO₂ comes mostly from burning fuel in engines; it is the standard marker for traffic exhaust,
the smell of the street. For each sensor, the median NO₂ on weekday mornings, 7 to 9 a.m., over a
full year (November 2025 to October 2026). A smooth surface spread from the sensors is drawn as
illuminated (Tanaka) contours and read at every CTA bus stop.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | Open Air hourly measurements, one file per month (cached); the sensor list |
| 3. Validate | Columns, window, sensors, collocated sensors |
| 4. Transform | Per sensor: medians at the morning rush and all day; hourly profiles |
| 5. Analyze | Surface, contours, CTA bus stops, sides, hours of the day |
| 6. QA | Collocated sensors agree; standards; PM2.5 |
| 7. Export | `plate.json` |

Run top to bottom from this folder. The per-sensor summary (`data/raw/openair_sensors_*.csv`) is
committed; the hourly files (about 10 MB a month) are cached in `data/processed/openair/` and only
needed with `REFRESH = True`.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import zipfile
from datetime import date
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from matplotlib.path import Path as MPath
from scipy.ndimage import map_coordinates
from scipy.spatial import cKDTree

from atlas import Ledger, write_plate
from atlas.boundary import simplify
from atlas.gtfs import service_ids
from atlas.grid import miles_from_origin

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day19')
pd.set_option('display.width', 140)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = download the hourly files again and rebuild the per-sensor summary

DAY = 19
DATASET = 'di9s-96ws'  # Open Air Chicago Hour Aggregations
DATASET_URL = 'https://data.cityofchicago.org/d/di9s-96ws'
SODA = 'https://data.cityofchicago.org/resource/di9s-96ws.csv'
GTFS_URL = 'https://www.transitchicago.com/developers/gtfs/'
WHO_URL = 'https://www.who.int/publications/i/item/9789240034228'
EPA_URL = 'https://www.epa.gov/criteria-air-pollutants/naaqs-table'
MONTHS = pd.period_range('2025-11', '2026-10', freq='M')  # twelve full months
PULLED = date(2026, 11, 22)
RUSH = (7, 8)            # local hours: 7:00-8:59 a.m., weekdays
MIN_HOURS = 24 * 120     # a sensor needs 120 days of readings
SIGMA = 0.6              # miles: Gaussian kernel of the surface
REACH = 1.25             # miles: the surface is drawn where the kernel weighs at least one sensor this far away
STEP = 1 / 32            # miles: surface grid
EXT = (-14, 6, -17, 13)  # miles west, east, south, north (the Plates II-XVIII extent)
WHO_PPB = 5.3            # WHO 2021 annual guideline, 10 ug/m3 NO2 at 25 C
EPA_PPB = 53             # US EPA annual standard (NAAQS), 53 ppb
HIGH = 20                # ppb: 'high' for the text

DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
RAW_DIR = DAY_DIR / 'data' / 'raw'
HOUR_DIR = DAY_DIR / 'data' / 'processed' / 'openair'
SUMMARY = RAW_DIR / f'openair_sensors_{PULLED.isoformat()}.csv'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
GTFS_ZIP = REPO / 'reference' / 'raw' / 'gtfs' / 'cta_gtfs_2026-11-08.zip'
SERVICE_DAY = date(2026, 11, 17)  # a Tuesday
BOUNDARY = REPO / 'reference' / 'raw' / 'city_boundary_2026-11-01.geojson'
BLOCKS = REPO / 'reference' / 'chicago-blocks-2020.csv'
assert (DAY_DIR / 'day19.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'
# Chicago's nine sides, by community area (as Plates X and XVII): Wikipedia's grouping (unofficial)
SIDES = {
    'Central': [8, 32, 33],
    'North Side': [5, 6, 7, 21, 22],
    'Far North Side': [1, 2, 3, 4, 9, 10, 11, 12, 13, 14, 76, 77],
    'Northwest Side': [15, 16, 17, 18, 19, 20],
    'West Side': [23, 24, 25, 26, 27, 28, 29, 30, 31],
    'South Side': [34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 60, 68, 69],
    'Southwest Side': [56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67],
    'Far Southwest Side': [70, 71, 72, 73, 74, 75],
    'Far Southeast Side': [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55],
}
SIDE_OF = {ca: s for s, cas in SIDES.items() for ca in cas}
assert sorted(SIDE_OF) == list(range(1, 78))

## 2. Pull
One CSV per month from the portal (UTC hours), four columns. Cached; the committed per-sensor summary is enough to run Sections 5 to 7.

In [ ]:
def month_file(p: pd.Period) -> Path:
    return HOUR_DIR / f'hour_{p}.csv'


def pull_month(p: pd.Period) -> None:
    start, end = p.start_time, (p + 1).start_time
    params = {'$select': 'datasourceid,startofperiod,no2conc1hourmean_value,pm2_5concmass1hourmean_value',
              '$where': f"startofperiod>='{start:%Y-%m-%dT%H:%M:%S}' and startofperiod<'{end:%Y-%m-%dT%H:%M:%S}'",
              '$order': 'datasourceid,startofperiod', '$limit': 2_000_000}
    r = requests.get(SODA, params=params, timeout=900)
    r.raise_for_status()
    month_file(p).write_bytes(r.content)


HOUR_DIR.mkdir(parents=True, exist_ok=True)
have_hours = all(month_file(p).exists() for p in MONTHS)
if REFRESH:
    for p in MONTHS:
        log.info('pulling %s', p)
        pull_month(p)
    have_hours = True
    r = requests.get(SODA, params={'$select': 'datasourceid,sensor_name,latitude,longitude,count(*) as n',
                                   '$group': 'datasourceid,sensor_name,latitude,longitude', '$limit': 5000}, timeout=300)
    r.raise_for_status()
    (HOUR_DIR / 'sensors.csv').write_bytes(r.content)
log.info('hourly files cached: %s; summary committed: %s', have_hours, SUMMARY.exists())

## 3. Validate

In [ ]:
if have_hours and (REFRESH or not SUMMARY.exists()):
    h = pd.concat([pd.read_csv(month_file(p)) for p in MONTHS], ignore_index=True)
    h.columns = ['id', 't', 'no2', 'pm']
    h['t'] = pd.to_datetime(h['t']).dt.tz_localize('UTC').dt.tz_convert('America/Chicago')
    h = h[(h.t >= MONTHS[0].start_time.tz_localize('America/Chicago')) & (h.t < (MONTHS[-1] + 1).start_time.tz_localize('America/Chicago'))]
    assert not h.duplicated(['id', 't']).any(), 'duplicate sensor-hours'
    assert h.no2.between(-5, 500).all() or h.no2.isna().any(), 'NO2 out of range'
    sensors = pd.read_csv(HOUR_DIR / 'sensors.csv')
    assert sensors.datasourceid.is_unique, 'a sensor reports from two places'
    log.info('%d hourly rows, %d sensors, %s to %s', len(h), h.id.nunique(), h.t.min(), h.t.max())
else:
    h = None
    log.info('using the committed summary %s', SUMMARY.name)

## 4. Transform
Per sensor: hours with an NO₂ reading; median NO₂ on weekday mornings (7:00 to 8:59 local) and over all hours; the annual mean (for the standards); PM2.5 the same way; the median of each hour of the day on weekdays and on weekends.

In [ ]:
if h is not None:
    h['wd'] = h.t.dt.dayofweek < 5
    h['hr'] = h.t.dt.hour
    rush = h[h.wd & h.hr.isin(RUSH)]
    g = h.groupby('id')
    S = pd.DataFrame({
        'hours': g.no2.count(), 'no2_rush': rush.groupby('id').no2.median(), 'no2_median': g.no2.median(),
        'no2_mean': g.no2.mean(), 'pm_rush': rush.groupby('id').pm.median(), 'pm_mean': g.pm.mean()})
    wd = h[h.wd].pivot_table(index='id', columns='hr', values='no2', aggfunc='median').add_prefix('wd')
    we = h[~h.wd].pivot_table(index='id', columns='hr', values='no2', aggfunc='median').add_prefix('we')
    S = S.join(sensors.set_index('datasourceid')[['sensor_name', 'latitude', 'longitude']]).join(wd).join(we)
    S.index.name = 'id'
    S.round(3).to_csv(SUMMARY)
S = pd.read_csv(SUMMARY, index_col='id')
S['collocation'] = S.sensor_name.str.contains('Collocation')
S['x'], S['y'] = miles_from_origin(S.latitude.values, S.longitude.values)
net = S[~S.collocation & (S.hours >= MIN_HOURS)].copy()
log.info('%d sensors (%d collocated, %d short of %d hours)', len(S), S.collocation.sum(),
         (~S.collocation & (S.hours < MIN_HOURS)).sum(), MIN_HOURS)
net[['no2_rush', 'no2_median', 'no2_mean', 'pm_rush']].describe().round(1)

## 5. Analyze
### 5a. The surface
A Gaussian kernel (σ = 0.6 mi) spreads the sensors' morning medians into a smooth surface. It is drawn inside the city where the kernel weighs at least as much as one sensor 1.25 miles away.

In [ ]:
x0, x1, y0, y1 = EXT
gx = np.arange(x0, x1, STEP) + STEP / 2
gy = np.arange(y0, y1, STEP) + STEP / 2
GX, GY = np.meshgrid(gx, gy)
num = np.zeros(GX.shape); den = np.zeros(GX.shape)
for x, y, v in zip(net.x, net.y, net.no2_rush):
    w = np.exp(-((GX - x) ** 2 + (GY - y) ** 2) / (2 * SIGMA ** 2))
    num += w * v
    den += w
Z = num / den

gj = json.loads(BOUNDARY.read_text())
P = np.column_stack([GX.ravel(), GY.ravel()])
city = np.zeros(GX.shape, bool)
for f in gj['features']:
    geom = f['geometry']
    for poly in (geom['coordinates'] if geom['type'] == 'MultiPolygon' else [geom['coordinates']]):
        for k, ring in enumerate(poly):
            r = np.asarray(ring)
            inside = MPath(np.column_stack(miles_from_origin(r[:, 1], r[:, 0]))).contains_points(P).reshape(GX.shape)
            city = (city | inside) if k == 0 else (city & ~inside)
drawn = city & (den >= np.exp(-REACH ** 2 / (2 * SIGMA ** 2)))
zs = Z[drawn]
LEVELS = np.arange(np.floor(zs.min()), np.ceil(zs.max()) + 1, 1.0)
log.info('surface %.1f to %.1f ppb; %.1f%% of the city drawn; %d levels', zs.min(), zs.max(), 100 * drawn.sum() / city.sum(), len(LEVELS))

### 5b. Contours
Filled bands between whole ppb, and the contour lines with their light: each short piece of a line is lit (white) where the slope beneath it faces the light in the northwest and shaded (ink) where it faces away, in nine steps (Tanaka's illuminated contours). Coordinates in thousandths of a mile.

In [ ]:
matplotlib.use('Agg')
Zd = np.where(drawn, Z, np.nan)
gyz, gxz = np.gradient(Z, STEP)
LIGHT = np.array([-1.0, 1.0]) / np.sqrt(2)  # toward the light: northwest
TOL = 0.004                                  # miles: simplification

def mm(a):
    return np.round(np.asarray(a) * 1000).astype(int).tolist()

cf = plt.contourf(GX, GY, Zd, levels=np.r_[LEVELS[0] - 5, LEVELS[1:-1], LEVELS[-1] + 5])
bands = []
for i, segs in enumerate(cf.allsegs):
    rings = [mm(simplify(s, TOL)) for s in segs if len(s) >= 4]
    bands.append({'lo': float(LEVELS[i]), 'rings': [r for r in rings if len(r) >= 4]})
plt.close()

cs = plt.contour(GX, GY, Zd, levels=LEVELS[1:-1])
lines = []
for lev, segs in zip(cs.levels, cs.allsegs):
    for s in segs:
        s = simplify(s, TOL)
        if len(s) < 2:
            continue
        m = (s[:-1] + s[1:]) / 2
        ii, jj = (m[:, 1] - gy[0]) / STEP, (m[:, 0] - gx[0]) / STEP
        grad = np.column_stack([map_coordinates(gxz, [ii, jj], order=1), map_coordinates(gyz, [ii, jj], order=1)])
        n = grad / (np.linalg.norm(grad, axis=1, keepdims=True) + 1e-12)
        lit = -(n @ LIGHT)                    # +1: the slope faces the light
        k = np.clip(np.round(lit * 4), -4, 4).astype(int) + 4
        lines.append({'lev': float(lev), 'pts': mm(s), 'lit': ''.join(map(str, k))})
plt.close()
log.info('%d bands (%d rings), %d lines (%d points)', len(bands), sum(len(b['rings']) for b in bands), len(lines), sum(len(l['pts']) for l in lines))

### 5c. Where riders wait
Every CTA bus stop served on a weekday (Tuesday 17 November 2026) inside the city, read on the surface; the buses due there 7 to 9 a.m.; community area and side from the nearest 2020 census block.

In [ ]:
z = zipfile.ZipFile(GTFS_ZIP)
stops = pd.read_csv(z.open('stops.txt'), dtype={'stop_id': str})
stops = stops[stops.location_type.fillna(0) == 0]
routes = pd.read_csv(z.open('routes.txt'), dtype=str)
bus = set(routes.loc[routes.route_type == '3', 'route_id'])
trips = pd.read_csv(z.open('trips.txt'), dtype=str)
trips = trips[trips.route_id.isin(bus) & trips.service_id.isin(service_ids(GTFS_ZIP, SERVICE_DAY))]
want = set(trips.trip_id)
st = pd.concat(ch[ch.trip_id.isin(want)] for ch in pd.read_csv(
    z.open('stop_times.txt'), dtype=str, chunksize=2_000_000, usecols=['trip_id', 'stop_id', 'departure_time']))
st['hr'] = st.departure_time.str[:2].astype(int)
served = st.groupby('stop_id').size()
rushdep = st[st.hr.isin(RUSH)].groupby('stop_id').size()
bs = stops[stops.stop_id.isin(served.index)].copy()
bs['x'], bs['y'] = miles_from_origin(bs.stop_lat.values, bs.stop_lon.values)
ii, jj = (bs.y - gy[0]) / STEP, (bs.x - gx[0]) / STEP
bs = bs[map_coordinates(city.astype(float), [ii, jj], order=0) > 0].copy()
ii, jj = (bs.y - gy[0]) / STEP, (bs.x - gx[0]) / STEP
bs['no2'] = map_coordinates(Z, [ii, jj], order=1)
bs['rush'] = bs.stop_id.map(rushdep).fillna(0)

blocks = pd.read_csv(BLOCKS)
bx, by = miles_from_origin(blocks.lat.values, blocks.lon.values)
_, near = cKDTree(np.column_stack([bx, by])).query(np.column_stack([bs.x, bs.y]))
bs['ca'] = blocks.commarea.values[near]
bs['side'] = bs.ca.map(SIDE_OF)
_, nearS = cKDTree(np.column_stack([bx, by])).query(np.column_stack([net.x, net.y]))
net['side'] = blocks.commarea.values[nearS]
net['side'] = net.side.map(SIDE_OF)
blocks['no2'] = map_coordinates(Z, [(by - gy[0]) / STEP, (bx - gx[0]) / STEP], order=1)

# per band: bus stops and share of the drawn city (bands are whole ppb; the end bands are open)
edges = np.r_[-np.inf, LEVELS[1:-1], np.inf]
for i, b in enumerate(bands):
    b['stops'] = int(((bs.no2 >= edges[i]) & (bs.no2 < edges[i + 1])).sum())
    b['area'] = round(float(((Zd >= edges[i]) & (Zd < edges[i + 1])).sum() / drawn.sum()), 4)
side = (bs.groupby('side').agg(stops=('no2', 'size'), median=('no2', 'median'))
        .join(net.groupby('side').agg(sensors=('no2_rush', 'size')))
        .sort_values('median', ascending=False))
log.info('%d bus stops in the city; median %.1f ppb', len(bs), bs.no2.median())
side.round(1)

### 5d. Hours of the day
For each hour, the middle sensor (median over sensors of each sensor's median for that hour), and the middle half (quartiles), on weekdays and on weekends.

In [ ]:
wdc = [f'wd{i}' for i in range(24)]; wec = [f'we{i}' for i in range(24)]
curve = {
    'weekday': {q: net[wdc].quantile(p).round(2).tolist() for q, p in (('p25', .25), ('p50', .5), ('p75', .75))},
    'weekend': {q: net[wec].quantile(p).round(2).tolist() for q, p in (('p25', .25), ('p50', .5), ('p75', .75))},
}
wd50 = np.array(curve['weekday']['p50'])
peak_h, low_h = int(wd50.argmax()), int(wd50.argmin())
log.info('weekday peak %02d:00 %.1f ppb; low %02d:00 %.1f ppb (+%.0f%%); weekend at the peak hour %.1f',
         peak_h, wd50[peak_h], low_h, wd50[low_h], 100 * (wd50[peak_h] / wd50[low_h] - 1), curve['weekend']['p50'][peak_h])

### 5e. Numbers for the text

In [ ]:
top = net.sort_values('no2_rush', ascending=False)
q = net.no2_rush.quantile([.1, .5, .9])
hi_band = HIGH
summary = {
    'sensors': int(len(net)), 'network': int((~S.collocation).sum()), 'collocated': int(S.collocation.sum()),
    'rushP10': round(float(q[.1]), 1), 'rushMedian': round(float(q[.5]), 1), 'rushP90': round(float(q[.9]), 1),
    'rushMax': round(float(top.no2_rush.iloc[0]), 1), 'rushMaxName': top.sensor_name.iloc[0],
    'rushMin': round(float(top.no2_rush.iloc[-1]), 1), 'rushMinName': top.sensor_name.iloc[-1],
    'surfaceMin': round(float(zs.min()), 1), 'surfaceMax': round(float(zs.max()), 1), 'drawnShare': round(float(drawn.sum() / city.sum()), 3),
    'peakHour': peak_h, 'lowHour': low_h, 'peak': round(float(wd50[peak_h]), 1), 'low': round(float(wd50[low_h]), 1),
    'weekendAtPeak': round(float(curve['weekend']['p50'][peak_h]), 1),
    'stops': int(len(bs)), 'stopsMedian': round(float(bs.no2.median()), 1),
    'stopsP10': round(float(bs.no2.quantile(.1)), 1), 'stopsP90': round(float(bs.no2.quantile(.9)), 1),
    'hiBand': float(hi_band), 'stopsHi': int((bs.no2 >= hi_band).sum()),
    'rushBusesHiShare': round(float(bs.rush[bs.no2 >= hi_band].sum() / bs.rush.sum()), 3),
    'stopsHiShare': round(float((bs.no2 >= hi_band).mean()), 3),
    'stopsHiSides': {s: int(n) for s, n in bs[bs.no2 >= hi_band].side.value_counts().items()},
    'residentsHi': int(blocks['pop'][blocks.no2 >= hi_band].sum()), 'residents': int(blocks['pop'].sum()),
    'meanMin': round(float(net.no2_mean.min()), 1), 'meanMax': round(float(net.no2_mean.max()), 1),
    'whoPpb': WHO_PPB, 'epaPpb': EPA_PPB,
    'pmRushP10': round(float(net.pm_rush.quantile(.1)), 1), 'pmRushP90': round(float(net.pm_rush.quantile(.9)), 1),
}
summary['sides'] = [{'side': s, 'stops': int(r.stops), 'median': round(float(r['median']), 1), 'sensors': int(r.sensors)} for s, r in side.iterrows()]
summary

## 6. QA
The nine collocated sensors stand three by three beside federal monitors (Mayfair, Schiller Park, Washington High School). Readings at one place should agree; their spread is the floor below which differences on the map mean nothing.

In [ ]:
col = S[S.collocation].copy()
col['site'] = col.sensor_name.str.replace(r' Collocation \d+$', '', regex=True)
spread = col.groupby('site').no2_rush.agg(lambda v: v.max() - v.min())
qa = {'collocSites': int(col.site.nunique()), 'collocSpreadMax': round(float(spread.max()), 1),
      'collocSpreadMedian': round(float(spread.median()), 1)}
if h is not None:
    X = {s: h[h.id.isin(g.index)].pivot_table(index='t', columns='id', values='no2') for s, g in col.groupby('site')}
    rs = [c for x in X.values() for c in x.corr().values[np.triu_indices(x.shape[1], 1)]]
    qa['collocHourlyR'] = [round(float(min(rs)), 2), round(float(max(rs)), 2)]
assert spread.max() < 5, 'collocated sensors disagree by 5 ppb or more'
assert net.no2_mean.max() < EPA_PPB and net.no2_mean.min() > WHO_PPB
assert summary['surfaceMax'] - summary['surfaceMin'] > 2 * qa['collocSpreadMedian'], 'map relief within the sensors\' own spread'
qa, spread.round(1).to_dict()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].contourf(GX, GY, Zd, levels=LEVELS, cmap='Greys'); ax[0].scatter(net.x, net.y, s=3, c='k'); ax[0].set_aspect('equal'); ax[0].set_title('Weekday 7-9 a.m. NO2 (ppb)')
for k, ls in (('weekday', '-'), ('weekend', '--')):
    ax[1].plot(curve[k]['p50'], ls, label=k); ax[1].fill_between(range(24), curve[k]['p25'], curve[k]['p75'], alpha=.15)
ax[1].legend(); ax[1].set_xlabel('hour'); ax[1].set_ylabel('ppb'); plt.show()

## 7. Export

In [ ]:
side_names = {s['side']: s for s in summary['sides']}
hiSides = [s for s in summary['sides'] if s['side'] in ('Southwest Side', 'West Side', 'Central')]
loSides = [s for s in summary['sides'] if s['side'] in ('Far North Side', 'Far Southwest Side', 'Far Southeast Side')]
up = summary['peak'] / summary['low'] - 1
hname = lambda hh: f"{(hh % 12) or 12} {'a.m.' if hh < 12 else 'p.m.'}"
finding = (f"The air at Chicago's bus stops is dirtiest in the morning rush. At {hname(summary['peakHour'])} on weekdays it holds "
           f"about {up:.0%} more nitrogen dioxide than at {hname(summary['lowHour'])}, the cleanest hour. The worst of it lies over "
           f"the Loop and the Southwest and West Sides.")
ledger = Ledger(
    source='Open Air Chicago (Chicago Department of Public Health; UIC School of Public Health), hour aggregations',
    dataset_id=DATASET, url=DATASET_URL, pulled=PULLED,
    units='NO2, parts per billion (Clarity cloud-calibrated)',
    method=['Median per sensor, weekdays 7-9 a.m. local, Nov 2025 - Oct 2026',
            f'Gaussian kernel surface, sigma {SIGMA} mi; drawn where it weighs one sensor at {REACH} mi',
            'Contours every 1 ppb, lit from the northwest (Tanaka)'],
    caveats=['Non-regulatory sensors; collocated sensors differ by up to ' + f"{qa['collocSpreadMax']} ppb",
             'A smoothed surface: values between sensors are estimates'],
)
cols = ['sensor_name', 'x', 'y', 'hours', 'no2_rush', 'no2_median', 'no2_mean', 'pm_rush', 'side']
payload = {
    'window': [str(MONTHS[0]), str(MONTHS[-1])],
    'levels': LEVELS.tolist(), 'sigma': SIGMA, 'reach': REACH, 'rush': list(RUSH),
    'gtfs': {'file': GTFS_ZIP.name, 'snapshot': GTFS_ZIP.stem.split('_')[-1], 'day': SERVICE_DAY.isoformat(), 'url': GTFS_URL},
    'sources': {'dataset': DATASET_URL, 'who': WHO_URL, 'epa': EPA_URL,
                'calibration': 'https://www.clarity.io/blog/introducing-claritys-global-no2-calibration-model'},
    'summary': summary, 'qa': qa, 'curve': curve,
    'sensors': [[r.sensor_name, round(r.x, 3), round(r.y, 3), int(r.hours), round(r.no2_rush, 1), round(r.no2_median, 1),
                 round(r.no2_mean, 1), round(r.pm_rush, 1), r.side] for r in net.sort_values('no2_rush')[cols].itertuples()],
    'sensorCols': cols,
    'bands': bands, 'lines': lines,
}
write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=finding, payload=payload)
log.info('Wrote %s (%.0f KB)', OUT_PATH, OUT_PATH.stat().st_size / 1024)
finding